In [26]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

В данной лабораторной работе Вам нужно будет реализовать Логистическую Регрессию с функцией потерь `Binary Cross Entropy` а так же метода стандартизации из `sklearn` `StandardScaler`

# Sigmoid

In [27]:
def sigmoid(x):
    x_clipped = np.clip(x, -250, 250)
    return 1.0 / (1.0 + np.exp(-x_clipped))

# StandardScaler (2 балла )

In [28]:
class StandardScaler:
    def __init__(self):
        self.mean_ = None
        self.var_ = None

    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.var_ = np.var(X, axis=0)
        return self

    def transform(self, X):
        std = np.sqrt(self.var_ + 1e-8)
        return (X - self.mean_) / std

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


# LogisticRegression (6 баллов)
## - `predict_proba` (0.5 балла)
## - `predict` (0.5 балла)
## - `loss` (1 балл )
$$
{L}_{\text{BCE}} = -\frac{1}{N}\sum_{i=1}^{N}\left[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \right]
$$

- $y_i \in \{0, 1\}$ — истинная метка класса
- $\hat{y}_i \in (0, 1)$ — предсказанная вероятность класса $1$
- $N$ — количество примеров
- $\log$ — натуральный логарифм

## - `loss_grad` (1 балл )
## - `step` (0.5 балла)
## - `fit` (2.5 балла)

In [29]:
class LogisticRegression:
    def __init__(self, lr=0.01, max_iter=100, batch_size=16, fit_intercept=True):
        self.lr = lr
        self.max_iter = max_iter
        self.batch_size = batch_size
        self.fit_intercept = fit_intercept
        self.coef_ = None

    def _add_intercept(self, X):
        if self.fit_intercept:
            return np.c_[X, np.ones((X.shape[0], 1))]
        return X

    def predict_proba(self, X):
        X_ext = self._add_intercept(X)
        logits = np.dot(X_ext, self.coef_)
        return sigmoid(logits)
        
    def predict(self, X):
        probs = self.predict_proba(X)
        return (probs >= 0.5).astype(int)

    def loss(self, y_true, probs):
        eps = 1e-15
        probs_clipped = np.clip(probs, eps, 1 - eps)
        y_true = y_true.flatten()
        probs_clipped = probs_clipped.flatten()
        return -np.mean(y_true * np.log(probs_clipped) + (1 - y_true) * np.log(1 - probs_clipped))

    def loss_grad(self, X, y_true):
        X_ext = self._add_intercept(X)
        probs = self.predict_proba(X).flatten()
        y_true = y_true.flatten()
        
        error = probs - y_true
        grad = np.dot(X_ext.T, error) / X.shape[0]
        return grad
        
    def step(self, grad):
        self.coef_ = self.coef_ - self.lr * grad

    def fit(self, X, y):
        n_features = X.shape[1]        
        if self.fit_intercept:
            self.coef_ = np.zeros(n_features + 1)
        else:
            self.coef_ = np.zeros(n_features)
            
        for _ in range(self.max_iter):
            grad = self.loss_grad(X, y)
            self.step(grad)
            
        return self
    

# Тест

Код ниже для проверки Вашего решения, его можно запускать для самопроверки, **но нельзя менять**

In [30]:
def test_sigm():
    x = np.array([0, 1, -1, 1e-9, 1e9])
    y_true = np.array([1, 0, 1])

    correct = np.array([0.5,        0.73105858, 0.26894142, 0.5,        1. ])
    actual = sigmoid(x)

    assert np.allclose(actual, correct,1e-6), "Sigmoid Test Failed"



def test_standard_scaler_1():
    scaler = StandardScaler()

    X = np.array([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])
    X_scaled = scaler.fit_transform(X)
    correct_X_scaled = np.array([
        [-1.22474487, -1.22474487, -1.22474487],
        [0.        ,  0.        ,  0.        ],
        [1.22474487,  1.22474487,  1.22474487]])

    assert np.allclose(X_scaled, correct_X_scaled,1e-6), "Standard Scaler Test 1 Failed"


def test_standard_scaler_2():
    scaler = StandardScaler()

    X = np.array([[1, 1],
                  [1, 1],
                  [1, 1]])
    scaler.fit(X)
    X_scaled = scaler.transform(X)
    correct_X_scaled = np.array([
        [0.0, 0.0],
        [0.0, 0.0],
        [0.0, 0.0]])

    assert np.allclose(X_scaled, correct_X_scaled,1e-6), "Standard Scaler Test 2 Failed"


def test_predict_proba_1():

    model = LogisticRegression()
    model.coef_ = np.array([8, 1])

    X = np.array([[-1.34783067],
       [-1.82698213],
       [-1.0395786],
       [-0.48313131],
       [ 0.54284994],
       [-2.31929004],
       [-0.19568926],
       [-2.4347564],
       [-0.76744448],
       [-2.9446212 ]])

    actual = model.predict_proba(X)

    correct = np.array([5.64191577e-05,
                        1.22097815e-06,
                        6.63957127e-04,
                        5.39085328e-02,
                        9.95240300e-01,
                        2.37823924e-08,
                        3.62272570e-01,
                        9.44244195e-09,
                        5.82614911e-03,
                        1.59820533e-10])

    assert np.allclose(actual,correct,1e-6), 'Predict Proba Test 1 Failed'


def test_predict_proba_2():

    model = LogisticRegression(fit_intercept=False)
    model.coef_ = np.array([8, 3, 2, -2])

    X = np.array([[-1.88410657, -1.4562849 , -0.26786086, -1.98337983],
       [ 0.49296843, -1.31316104, -1.33452649, -1.12234877],
       [ 0.17948176, -1.19334198, -1.02785634, -2.0610986 ],
       [-0.42685327, -1.73920213, -1.04990344, -3.48129608],
       [-1.04109474,  0.0756424 , -0.13264918, -1.58288797],
       [-2.3213775 , -3.44956942, -1.80228679,  0.28671583],
       [-1.83510376, -0.56379521, -2.11184355, -0.99271932],
       [-1.81245323, -0.29125698, -2.23863083, -2.37841275],
       [-0.64343871, -1.90509213, -0.55792398,  0.14948942],
       [ 0.68655732, -2.49724917, -1.50467435, -1.32191236]])

    actual = model.predict_proba(X)

    correct = np.array([1.11340871e-07,
                        3.96494946e-01,
                        4.80587927e-01,
                        2.25404096e-02,
                        5.47892084e-03,
                        4.22509986e-15,
                        8.27138244e-09,
                        2.78504055e-07,
                        4.65477248e-06,
                        8.58955413e-02])

    assert np.allclose(actual,correct,1e-6), 'Predict Proba Test 2 Failed'


def test_predict_1():

    model = LogisticRegression()
    model.coef_ = np.array([5, 1])

    X = np.array([[-1.34783067],
       [-1.82698213],
       [-1.0395786],
       [-0.48313131],
       [ 0.54284994],
       [-2.31929004],
       [-0.19568926],
       [-2.4347564 ],
       [-0.76744448],
       [-2.9446212 ]])

    actual = model.predict(X)

    correct = np.array([0, 0, 0, 0, 1, 0, 1, 0, 0, 0])

    assert np.allclose(actual,correct,1e-6), 'Predict Test 1 Failed'


def test_predict_2():

    model = LogisticRegression(fit_intercept=False)
    model.coef_ = np.array(
        [15, 0.5, -2])

    X = np.array([
       [-1.88410657, -1.4562849 , -0.26786086],
       [ 0.49296843, -1.31316104, -1.33452649],
       [ 0.17948176, -1.19334198, -1.02785634 ],
       [-0.42685327, -1.73920213, -1.04990344],
       [-1.04109474,  0.0756424 , -0.13264918],
       [-2.3213775 , -3.44956942, -1.80228679],
       [-1.83510376, -0.56379521, -2.11184355],
       [-1.81245323, -0.29125698, -2.23863083],
       [-0.64343871, -1.90509213, -0.55792398],
       [ 0.68655732, -2.49724917, -1.50467435]])

    actual = model.predict(X)

    correct = np.array([0, 1, 1, 0, 0, 0, 0, 0, 0, 1])

    assert np.allclose(actual,correct,1e-6), 'Predict Test 2 Failed'

def test_loss():

    model = LogisticRegression()

    y_true = np.array([1, 0, 1])
    probs = np.array(
        [0.7,0.3,0.1])

    correct = 1.005311659861932
    actual = model.loss(y_true, probs)

    assert np.isclose(actual, correct), "Loss Test Failed"


def test_step_1():

    model = LogisticRegression(lr=0.1, fit_intercept=True)
    model.coef_ = np.array(
        [8, 3, 2, 5], dtype=np.float64)

    loss_grad = np.array(
        [-0.53333333,  0.16666667,  0.1, 0.31], dtype=np.float64)

    correct_coef = np.array(
        [8.05333333, 2.98333333, 1.99,       4.969     ], dtype=np.float64)

    model.step(loss_grad)
    assert np.allclose(model.coef_,correct_coef,1e-6), 'Step Test 1 Failed: incorrect coef'

def test_step_2():

    model = LogisticRegression(lr=0.1, fit_intercept=False)
    model.coef_ = np.array(
        [8, 3, 2], dtype=np.float64)

    loss_grad = np.array(
        [-0.53333333,  0.16666667,  0.1,], dtype=np.float64)

    correct_coef = np.array(
        [8.05333333, 2.98333333, 1.99], dtype=np.float64)

    model.step(loss_grad)
    assert np.allclose(model.coef_,correct_coef,1e-6), 'Step Test 2 Failed: incorrect coef'



def test_grad_1():

    model = LogisticRegression(fit_intercept=False)
    model.coef_ = np.array([8, 3, 2])

    y_true = np.array([0, 1, 1])
    X = np.array([
        [0.7, 0.2, 0.1],
        [0.3, 0.5, 0.2],
        [0.1, 0.3, 0.6]
    ])

    correct = np.array([0.22986914, 0.05910957, 0.02195487])

    actual = model.loss_grad(X, y_true)

    assert np.allclose(actual,correct,1e-6), 'Grad Test 1 Failed: incorrect grad'

def test_grad_2():

    model = LogisticRegression(fit_intercept=True)
    model.coef_ = np.array([8, 3, 2, 5])

    y_true = np.array([0, 1, 1])
    X = np.array([
        [0.7, 0.2, 0.1],
        [0.3, 0.5, 0.2],
        [0.1, 0.3, 0.6]
    ])

    correct = np.array([0.23330923, 0.06661362, 0.03325274, 0.33317559])

    actual = model.loss_grad(X, y_true)

    assert np.allclose(actual,correct,1e-6), 'Grad Test 2 Failed: incorrect grad'

def test_fit():
    model = LogisticRegression(lr=0.1, max_iter=100)

    X, y = make_classification(n_samples=1000,
                               n_features=3,
                               n_informative=3,
                               n_redundant=0,
                               n_classes=2,
                               flip_y=0.07,
                               class_sep=2,
                               random_state=42)


    X, X_test, y, y_test = X[:100], X[100:], y[:100], y[100:]

    model.fit(X, y)

    acc = np.mean(model.predict(X_test) == y_test)

    assert acc >= 0.95, f'{acc} < 0.95'

test_sigm()
test_standard_scaler_1()
test_standard_scaler_2()
test_predict_proba_1()
test_predict_proba_2()
test_predict_1()
test_predict_2()
test_loss()
test_grad_1()
test_grad_2()
test_step_1()
test_step_2()
test_fit()

# Задача классификации (2 балла)
Необходимо:

1) Разбить данный датасет на `test` и `train` выделив 30% под `test`

2) Обучить написанную Вами модель логистической регресии

3) Посчитать метрику `Accuracy` (отношение числа верных ответов к общему числу ответов)


0.5 < `Accuracy` < 0.7 - 1 балл

0.7 < `Accuracy` < 1.0 - 2 балла

In [31]:
X = np.load('X.npy')
y = np.load('Y.npy')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(lr=0.1, max_iter=500, fit_intercept=True)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
accuracy = np.mean(y_pred == y_test)

print(f"Accuracy на тестовой выборке: {accuracy:.4f}")

Accuracy на тестовой выборке: 0.9258
